# Reconstructing one missing example

Our main attack methodology is designed to reconstruct the entire training set of a given tree ensemble. However, most prior works tackling reconstruction attacks consider a simpler setup where all but one of the training examples are known, and the attacker’s objective is to reconstruct the remaining example. As in our paper’s appendices, we therefore also provide a specific version of our attack tackling this special case.

An informed adversary knows **all training examples and labels except one** and observes a DP-protected random forest. Here we reconstruct that one missing feature vector, with the same COMPAS data and DP random forest settings as the main demo.

Install with `python -m pip install -e ".[notebook]"` and open this notebook from the repository root. The saved results below can produce different timings and solver outcomes.

In [1]:
from copy import deepcopy
import numpy as np
import pandas as pd
from IPython.display import display
from DP_RF import DP_RF
from DP_RF_solver import DRAFT_DP
from datasets_infos import datasets_ohe_vectors, predictions
from utils import load_dataset, display_dataset, print_reconstruction_results

seed = 0
target_index = 0  # zero-based row position, not a pandas index label
epsilons = [10, 5]
options = dict(timeout=60, n_threads=2, seed=seed)
ohe_vector = datasets_ohe_vectors["compas"]
X_train, X_test, y_train, y_test = load_dataset(
    "data/compas.csv", predictions["compas"],
    train_size=100, test_size=100, seed=seed,
)
print(f"{len(X_train)} training examples, {X_train.shape[1]} binary features.")

100 training examples, 14 binary features.


## What the adversary knows

The attack receives **only the 99 known feature vectors and their labels** through `X_known` and `y_known`.

This API supports **exactly one missing example**. It infers the total size as `len(X_known) + 1`. If you explicitly supply `n_samples`, it must equal that value. All supplied examples must be fully known, including their labels.

Here `target_index` is used only to simulate removing one training row and to evaluate the reconstructed target. It is not an attack argument.

In [2]:
known_rows = np.arange(len(X_train)) != target_index
known_X = X_train.to_numpy()[known_rows].copy()
known_y = y_train.to_numpy()[known_rows].copy()

display_dataset(
    X_train.iloc[np.flatnonzero(known_rows)[:8]],
    title="Adversary knowledge: first 8 known rows (target row 0 is absent)",
)
print(f"Only {len(known_X)} known feature vectors and labels are supplied.")
print("The missing example and label are absent from the attack inputs.")

Example,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,1,0,0,0,0,1,0,0,0,1
2,1,0,0,0,1,0,0,0,0,0,0,1,0,1
3,1,0,0,0,1,0,0,0,0,0,0,0,1,0
4,1,0,0,0,1,0,0,0,0,0,1,0,0,1
5,1,0,1,0,0,0,0,0,0,1,0,0,0,1
6,1,0,0,1,0,0,0,0,0,0,1,0,0,0
7,1,0,1,0,0,0,0,0,0,1,0,0,0,0
8,0,0,0,0,1,0,0,0,0,0,1,0,0,0


Only 99 known feature vectors and labels are supplied.
The missing example and label are absent from the attack inputs.


## Protect the random forest, then reconstruct

We train a 10-tree, depth-5 random-structure random forest and add noise to a fresh copy for each privacy budget.

The opt-in call supplies `X_known` and `y_known`. The sample count is inferred as the number of known rows plus one. The objective combines noisy-count likelihood (as in the whole DRAFT-DP formulation) and proximity to known examples, with a default scaling `target_ratio=epsilon / 2`.

For standard reconstruction instead, simply omit all three informed inputs:

```python
result = DRAFT_DP(protected_forest, epsilon, one_hot_encoded_groups=ohe_vector).fit(
    n_samples=len(X_train), **options,
)
```

The next cell uses the informed mode only.

In [3]:
base_forest = DP_RF(n_estimators=10, max_depth=5, random_state=seed, seed_noise=seed)
base_forest.fit(X_train, y_train)
results = {}
summary = []

for epsilon in epsilons:
    protected_forest = deepcopy(base_forest)
    protected_forest.add_noise(epsilon)
    result = DRAFT_DP(
        protected_forest, epsilon, one_hot_encoded_groups=ohe_vector,
    ).fit(
        X_known=known_X,
        y_known=known_y,
        **options,
    )
    results[epsilon] = result
    print(f"\nEpsilon {epsilon}: solver status {result['status']}")
    recovered = result["reconstructed_data"]
    if recovered is None:
        summary.append(dict(epsilon=epsilon, status=result["status"],
                            correct_features=None, target_error=None, exact_recovery=None,
                            duration_seconds=round(result["duration"], 3)))
        print_reconstruction_results("Informed adversary", result, X_train)
        continue

    recovered = np.asarray(recovered)
    assert np.array_equal(recovered[:-1], known_X)
    target = np.asarray(result["missing_example"])
    truth = X_train.iloc[target_index].to_numpy()  # evaluation only
    correct = int(np.sum(target == truth))
    error = float(np.mean(target != truth))
    summary.append(dict(epsilon=epsilon, status=result["status"],
                        correct_features=f"{correct}/{len(truth)}", target_error=error,
                        exact_recovery=bool(np.array_equal(target, truth)),
                        duration_seconds=round(result["duration"], 3)))
    print(f"Target row {target_index}: {correct}/{len(truth)} features correct.")
    print("The 99 known feature vectors are preserved.")
    # Evaluate just the unknown row; including fixed rows would inflate accuracy.
    target_result = {**result, "reconstructed_data": [target.tolist()]}
    print_reconstruction_results(
        f"DP informed target (epsilon {epsilon})", target_result,
        X_train.iloc[[target_index]], display_reconstruction=True,
    )


Epsilon 10: solver status OPTIMAL
Target row 0: 10/14 features correct.
The 99 known feature vectors are preserved.
(DP informed target (epsilon 10)) Complete solving duration : 0.299 seconds.
(DP informed target (epsilon 10)) Reconstruction Error: 0.285714


X_train row,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,1,0,0,0,0,0,0,0,1,0



Epsilon 5: solver status OPTIMAL
Target row 0: 10/14 features correct.
The 99 known feature vectors are preserved.
(DP informed target (epsilon 5)) Complete solving duration : 0.358 seconds.
(DP informed target (epsilon 5)) Reconstruction Error: 0.285714


X_train row,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,1,0,0,0,0,0,0,0,1,0
